# Listing photos and views: Shirley-Williams test

**Kruskal-Wallis Test with Multiple Comparisons** · a worked example from the StatNex teaching guide

**The story.** A resale app looked at 21 listings for the same kind of jacket, posted with 1, 3, 5 or 10 photos, and counted how many people viewed each listing in its first week.

**The question.** Do listings with more photos get more views, and how many photos are enough?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/kruskal_wallis.csv` (6 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
1 photo,3 photos,5 photos,10 photos
88.0,118.0,135.0,148.0
102.0,131.0,150.0,170.0
95.0,109.0,128.0,139.0
120.0,140.0,162.0,165.0
76.0,,141.0,158.0
110.0,,155.0,

These cells save `williams.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile williams.py
"""Williams' (1971, 1972) test for a dose response that only goes one way.

No Python library has it. Two pieces are written out: the isotonic estimate of each
dose's mean, and the critical value. Williams' printed table assumes equal dose groups;
this computes the value for the group sizes the design has -- the same numerical recipe
StatNex uses, so the constants agree with the app's to the last digit.

How: under the null hypothesis the differences D_u = (pooled mean of doses u..j) -
(control mean) have the covariance of one Brownian motion read at the times
1/n_0 + 1/N_u, N_u being the doses u..j pooled. The statistic stays below c exactly
when that motion stays below a line, a chance carried back from the last time to the
first on a grid, one convolution per dose. R's mvtnorm computes the same probability
with Miwa's and Genz's algorithms (pmvnorm, pmvt) and agrees to within 4e-6.
"""

import functools

import numpy as np
from scipy import signal, special, stats

# The grid: distance below the line, in standard errors, up to REACH. Its step is GRID, or an
# eighth of the walk's smallest step where that is finer (down to FINEST): a small step
# rises from 0 within a few sd of the line, and a coarser grid draws that as a straight line.
GRID, FINEST, REACH = 1e-3, 1e-4, 10.0


def isotonic_extreme(sizes, means, step, increasing):
    """The largest (or smallest) pooled mean of any block of doses ending at ``step``.

    ``sizes`` and ``means`` are ordered control first, then the doses.
    """
    pooled = [
        float((sizes[u:step + 1] * means[u:step + 1]).sum() / sizes[u:step + 1].sum())
        for u in range(1, step + 1)
    ]
    return max(pooled) if increasing else min(pooled)


@functools.lru_cache(maxsize=8)
def _scale_nodes(nu):
    """Gauss-Legendre nodes for E[...] over U = S/sigma, where nu*U^2 ~ chi-squared(nu)."""
    lo = np.sqrt(stats.chi2.ppf(1e-13, nu) / nu)
    hi = np.sqrt(stats.chi2.isf(1e-13, nu) / nu)
    x, w = special.roots_legendre(120)
    u = 0.5 * (hi - lo) * x + 0.5 * (hi + lo)
    log_f = ((nu / 2 - 1) * np.log(nu * u**2) - nu * u**2 / 2 - (nu / 2) * np.log(2.0)
             - special.gammaln(nu / 2) + np.log(2 * nu * u))
    wu = w * 0.5 * (hi - lo) * np.exp(log_f)
    return u, wu / wu.sum()


def _ramp(x, sd):
    """E[max(x - D, 0)] for D ~ N(0, sd^2): the normal CDF integrated once."""
    return x * stats.norm.cdf(x / sd) + sd * stats.norm.pdf(x / sd)


def _below_line(q, sd, d):
    """E[Q(h - D)] for D ~ N(0, sd^2), at h = 0, d, 2d, ...

    Q is straight lines between its nodes, 0 below h = 0 and 1 past the last node; each
    node's tent is integrated against the normal density exactly.
    """
    half = int(np.ceil(9.0 * sd / d)) + 2
    offsets = np.arange(-half, half + 1) * d
    tent = (_ramp(offsets + d, sd) - 2.0 * _ramp(offsets, sd) + _ramp(offsets - d, sd)) / d
    padded = np.concatenate([np.zeros(half), q, np.ones(half)])
    smoothed = signal.fftconvolve(padded, tent, mode="same")[half:half + q.size]
    # The node at h = 0 has no left half: Q is 0 just below it, not a ramp up to q[0].
    h = np.arange(q.size) * d
    left_half = (_ramp(h + d, sd) - _ramp(h, sd)) / d - stats.norm.cdf(h / sd)
    return smoothed - q[0] * left_half


@functools.lru_cache(maxsize=64)
def _curve(sizes):
    """P(statistic <= b) at infinite df, for b = 0, d, 2d, ..., and the grid step d.

    ``sizes`` is (control, dose 1, ..., the dose compared).
    """
    control = sizes[0]
    doses = np.asarray(sizes[1:], float)
    pooled = np.cumsum(doses[::-1])[::-1]
    times = (1.0 / control + 1.0 / pooled) / (1.0 / control + 1.0 / doses[-1])
    sds = np.sqrt(np.diff(times, prepend=0.0))   # sqrt(t_1), sqrt(t_2 - t_1), ...
    d = max(FINEST, min(GRID, float(sds.min()) / 8.0))
    q = np.ones(int(round(REACH / d)) + 1)
    for u in range(doses.size - 1, 0, -1):
        q = _below_line(q, float(sds[u]), d)
    return _below_line(q, float(sds[0]), d), d


def critical_value(sizes, step, nu, alpha):
    """Williams' constant for comparing dose ``step`` with the control.

    ``sizes`` are the group sizes, control first; ``nu`` the error degrees of freedom
    (``np.inf`` for the rank form, Shirley-Williams).
    """
    curve, d = _curve(tuple(float(s) for s in sizes[:step + 1]))
    grid = np.arange(curve.size) * d
    u, wu = (np.array([1.0]), np.array([1.0])) if np.isinf(nu) else _scale_nodes(float(nu))

    def tail(c):
        return 1.0 - float(wu @ np.interp(c * u, grid, curve, right=1.0))

    lo, hi = 0.0, 8.0
    while tail(hi) > alpha:   # very small nu: the one-dose 2.5% point at nu = 1 is 12.7
        lo, hi = hi, 2.0 * hi
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if tail(mid) > alpha else (lo, mid)
    return 0.5 * (lo + hi)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from williams import critical_value, isotonic_extreme

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per level. Text and blank cells are skipped, so levels may differ in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["1 photo", "3 photos", "5 photos", "10 photos"]
groups = [pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float) for name in levels]
sizes = np.array([len(g) for g in groups])

## Summary by level

In [ ]:
# All values are ranked together (ties share the average rank); each level's mean rank
# is what the test compares.
ranks = stats.rankdata(np.concatenate(groups))
mean_ranks = [part.mean() for part in np.split(ranks, np.cumsum(sizes)[:-1])]
summary = pd.DataFrame({"n": sizes, "mean rank": mean_ranks}, index=levels).T
print(summary)

## Kruskal-Wallis test

In [ ]:
# SciPy's H carries the tie correction; it is compared with chi-square on k - 1 df.
h, p = stats.kruskal(*groups)
df = len(groups) - 1
print(f"\nchi-square = {h:.6f}, df = {df}, P = {p:.6g}")

## Effect sizes

In [ ]:
# Both from the tie-corrected H (Tomczak & Tomczak 2014): epsilon squared H(n + 1)/(n^2 - 1)
# and eta squared H, (H - k + 1)/(n - k), which is below 0 when H is below its df. Both
# are read on Cohen's (1988) scale for eta squared: 0.01 small, 0.06 medium, 0.14 large.
n_total, k = sizes.sum(), len(groups)
effect_sizes = pd.Series({"epsilon squared": h * (n_total + 1) / (n_total**2 - 1),
                          "eta squared H": (h - k + 1) / (n_total - k) if n_total > k else np.nan})
print()
print(effect_sizes)

## Multiple comparison: Shirley-Williams

In [ ]:
# The first level is the control, and the rest are doses in the order given. Each dose is
# compared with the control assuming the response moves one way as the dose rises.
# Step-down: comparing dose i ranks only the control and doses 1..i afresh. A dose's
# mean rank is replaced by its isotonic estimate, and the statistic is compared with
# Williams' critical value at infinite df (the rank statistic is asymptotically normal),
# computed for the actual group sizes as the app does (williams.py); * at 5%, ** at 2.5%
# (one-sided), stepping down from the top dose: once a dose is not significant at a level,
# no lower dose is marked at it. Williams' printed table assumes equal groups, and his
# (1972) correction, which R's PMCMRplus applies at 5%, approximates the values for
# unequal ones.
all_sizes = [len(g) for g in groups]
rows = []
for label, increasing in (("control ≤", True), ("control ≥", False)):
    open5 = open25 = True
    for step in range(len(groups) - 1, 0, -1):
        subset = groups[: step + 1]
        sub_sizes = np.array([len(g) for g in subset], dtype=float)
        sub_ranks = stats.rankdata(np.concatenate(subset))
        sub_means = np.array([part.mean() for part in
                              np.split(sub_ranks, np.cumsum([len(g) for g in subset])[:-1])])
        total = int(sub_sizes.sum())
        _, tie_sizes = np.unique(sub_ranks, return_counts=True)
        variance = total * (total + 1) / 12.0 - float(np.sum(tie_sizes**3 - tie_sizes)) / (12.0 * (total - 1))
        estimate = isotonic_extreme(sub_sizes, sub_means, step, increasing)
        se = np.sqrt(variance * (1 / sub_sizes[step] + 1 / sub_sizes[0]))
        statistic = (estimate - sub_means[0]) / se if increasing else (sub_means[0] - estimate) / se
        crit5, crit25 = (critical_value(all_sizes, step, np.inf, a) for a in (0.05, 0.025))
        open25 = open25 and statistic >= crit25
        open5 = open5 and statistic >= crit5
        mark = "**" if open25 else "*" if open5 else ""
        rows.append([label, levels[0], levels[step], statistic, crit5, crit25, mark])
pairs = pd.DataFrame(rows, columns=["alternative", "level 1", "level 2", "statistic",
                                    "critical 5%", "critical 2.5%", ""])
print()
print(pairs.to_string(index=False))

## What the result says

The Kruskal-Wallis test compares groups by ranks: it sorts all the listings by views and compares each group's mean rank. The mean rank rises from 4 with 1 photo to 8.75, 14.33 and 17.2 with 3, 5 and 10 photos. The chi-square statistic is 14.89 and the P value, the chance of gaps this large if photos made no difference, is 0.002. Epsilon squared, a measure of effect size, is 0.74, which counts as large.

The Shirley-Williams test then compares each photo count with the control, 1 photo. It assumes that views do not fall as photos rise. A statistic above its critical value is significant. For 10 photos it is 3.51 and for 5 photos 3.27, both far above. For 3 photos it is 1.92: above the 5% critical value of 1.64, but below the stricter 2.5% value of 1.96.

So five or ten photos clearly beat one. Three photos also beat one at the 5% level, but only just. This test does not compare ten photos with five.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.